# Entraînement du classifieur mammo bénin/malin — Colab GPU (source **HF dataset**)

Pipeline : dataset HF privé → manifestes (split par patient stratifié) → entraînement
(`cropped` **et** `full`) → comparaison → publication de l'artefact versionné.

**Cibles** : AUC ≥ 0,85 · sensibilité ≥ 0,90 · spécificité ≥ 0,75. Runtime **GPU** requis.

> Le modèle de production est le **cropped** (le `ml-service` classe des ROIs YOLO).

## 1. Paramètres

In [ ]:
HF_DATASET_REPO = 'Mailcoding/ifar-mammo-rois'      # repo dataset HF privé (images + CSV)
HF_MODEL_REPO   = 'Mailcoding/ifar-mammo-classifier'
VERSION = 'v0.1.0'
CASE_CSVS = [
    'mass_case_description_train_set.csv',
    'calc_case_description_train_set.csv',
    'mass_case_description_test_set.csv',
    'calc_case_description_test_set.csv',
]
DICOM_INFO = 'dicom_info.csv'   # None si arbre DICOM->jpeg préservé
USES = ['cropped', 'full']
VAL_FRAC, SEED = 0.2, 42

## 2. Installer `ifar-mlops` (clone auto)

In [ ]:
# ifar-mlops est peuplé → on le clone. Repo PUBLIC : direct. Repo PRIVÉ : mets un PAT dans l'URL.
!git clone https://github.com/mailcoding/ifar-mlops.git /content/ifar-mlops 2>/dev/null || true
MLOPS_SRC = '/content/ifar-mlops'   # racine du repo (pyproject.toml + src/mlops)
import os
assert os.path.exists(f'{MLOPS_SRC}/pyproject.toml'), (
    f'{MLOPS_SRC}/pyproject.toml introuvable. Repo privé ? clone avec un PAT :\n'
    '  !git clone https://<PAT>@github.com/mailcoding/ifar-mlops.git /content/ifar-mlops')
!pip install "{MLOPS_SRC}[train]" -q
%cd "{MLOPS_SRC}"
import mlops, torch
print('mlops OK —', mlops.__file__, '| CUDA', torch.cuda.is_available())

## 3. Authentification Hugging Face

In [ ]:
import os
from huggingface_hub import login
os.environ['HF_TOKEN'] = 'hf_xxx'   # ← token HF (read dataset / write publication)
login(os.environ['HF_TOKEN'])

## 4. Télécharger le dataset (snapshot du repo HF privé)

In [ ]:
from huggingface_hub import snapshot_download
DATA_DIR = snapshot_download(repo_id=HF_DATASET_REPO, repo_type='dataset',
                             local_dir='/content/data')
print('Dataset local :', DATA_DIR)
!ls -la {DATA_DIR} | head

## 5. Générer les manifestes (split par patient stratifié) + class_weights

In [ ]:
import os, csv
from mlops.datasets.build_cbis_manifest import build

def write_manifest(rows, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, 'w', newline='') as f:
        w = csv.writer(f); w.writerow(['path', 'label'])
        for r in rows: w.writerow([r['path'], r['label']])

case_paths = [os.path.join(DATA_DIR, c) for c in CASE_CSVS]
dicom_info = os.path.join(DATA_DIR, DICOM_INFO) if DICOM_INFO else None
REPORTS = {}
for use in USES:
    rep = build(case_paths, use=use, dicom_info=dicom_info, images_root=DATA_DIR,
                val_frac=VAL_FRAC, seed=SEED, verify=True)
    s = rep['stats']
    assert not s['patient_leak'], f'FUITE PATIENT ({use}) !'
    if not (rep['train'] and rep['val']):
        print('  Exemples de chemins essayés (introuvables) :')
        for ex in s.get('example_missing', []): print('    -', ex)
    assert rep['train'] and rep['val'], (f"[{use}] train/val vide — vérifie DICOM_INFO/DATA_DIR "
                                         f"(résolues {s['n_resolved']}, manquants {s['missing_files']}).")
    write_manifest(rep['train'], f'/content/manifests/{use}/train.csv')
    write_manifest(rep['val'],   f'/content/manifests/{use}/val.csv')
    REPORTS[use] = rep
    print(f"[{use}] résolues={s['n_resolved']} (manquants {s['missing_files']}) | "
          f"train {s['train']} / val {s['val']} | class_weights {s['suggested_class_weights']}")

## 6. Entraîner les deux variantes (`cropped` puis `full`)

In [ ]:
import yaml, copy, subprocess
base = yaml.safe_load(open('configs/mammo_classifier.yaml'))
for use in USES:
    cfg = copy.deepcopy(base)
    cfg['data']['train_manifest'] = f'/content/manifests/{use}/train.csv'
    cfg['data']['val_manifest']   = f'/content/manifests/{use}/val.csv'
    cfg['train']['class_weights'] = REPORTS[use]['stats']['suggested_class_weights']
    cfg['export']['out_dir'] = f'/content/artifacts/{use}'
    cfg['export']['version'] = VERSION
    p = f'/content/config_{use}.yaml'; yaml.safe_dump(cfg, open(p, 'w'))
    print(f'\n===== ENTRAÎNEMENT [{use}] (class_weights={cfg["train"]["class_weights"]}) =====')
    subprocess.run(['python', '-m', 'mlops.train.train_mammo_classifier', '--config', p], check=True)

## 7. Comparer les métriques

In [ ]:
import json
for use in USES:
    m = json.load(open(f'/content/artifacts/{use}/manifest.json'))['metrics']
    print(f"[{use:7}] AUC={m.get('auc')} sens={m.get('sensitivity')} spec={m.get('specificity')} "
          f"| @sens>={m.get('sensitivity_target')}: thr={m.get('operating_threshold_sens')} "
          f"spec={m.get('specificity_at_target_sens')}")
print('\nRappel cibles : AUC>=0.85, sensibilité>=0.90, spécificité>=0.75.')

## 8. Publier l'artefact de production (variante `cropped`)

In [ ]:
import os, json
from mlops.registry import publish_artifact

PROD = 'cropped'
m = json.load(open(f'/content/artifacts/{PROD}/manifest.json'))['metrics']
ok = (m.get('auc') or 0) >= 0.85 and (m.get('sensitivity') or 0) >= 0.90
if not ok:
    print(f'⛔ Cibles non atteintes ({PROD}) — publication ANNULÉE. metrics={m}')
else:
    url = publish_artifact(f'/content/artifacts/{PROD}', HF_MODEL_REPO, VERSION)
    print('✅ Publié :', url, '(tag', VERSION, ')')

## Suite (hors notebook)
- **Porte de validation clinique** (SaMD) avant `validated: true` dans `manifest.json`.
- **Consommation produit** : le Space `ml-service` récupère la version (`pull_artifact` /
  `deploy-ml-space.yml`), nom de fichier `efficientnet_b0.pth`.
- ⚠️ Vider les sorties du notebook avant tout commit (aucune donnée patient dans git).